# Handwritten Character Recognition (A-Z)

**Author:** Adossi Fred William | CodeAlpha Machine Learning Internship

A convolutional neural network that recognises handwritten English capital
letters (A-Z). The notebook goes through loading and exploring the data,
preprocessing with augmentation, building a CNN with batch normalisation and
dropout, evaluating it (classification report and confusion matrix), and saving
the model for the HandScript AI Streamlit app.

---

## 1. Imports & Setup

Import the libraries, set up a dark theme for the plots, and fix the random
seeds so the results come out the same on each run.

In [ ]:
# --- Core scientific stack ---
import os
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# --- Deep learning ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Flatten, Dense,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# --- Machine learning utilities ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")
print("TensorFlow version:", tf.__version__)
print("GPU available     :", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
# --- Reproducibility: seed every random source ---
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Global dark plotting theme (facecolor #0F0F1A, axes #1A1A2E) ---
DARK_BG = "#0F0F1A"   # figure background
AXES_BG = "#1A1A2E"   # axes background
ACCENT  = "#00E5FF"   # primary accent (cyan)
ACCENT2 = "#FF4D6D"   # secondary accent (rose)
TEXT_C  = "#FFFFFF"

plt.rcParams.update({
    "figure.facecolor":  DARK_BG,
    "axes.facecolor":    AXES_BG,
    "savefig.facecolor": DARK_BG,
    "axes.edgecolor":    "#3A3A5A",
    "axes.labelcolor":   TEXT_C,
    "axes.titlecolor":   TEXT_C,
    "text.color":        TEXT_C,
    "xtick.color":       TEXT_C,
    "ytick.color":       TEXT_C,
    "grid.color":        "#2A2A40",
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "font.size":         11,
    "figure.dpi":        110,
})
sns.set_style("dark", {"axes.facecolor": AXES_BG})

# Label <-> letter helpers (0 -> 'A', ... 25 -> 'Z')
NUM_CLASSES = 26
LABELS = {i: chr(65 + i) for i in range(NUM_CLASSES)}
LETTERS = [LABELS[i] for i in range(NUM_CLASSES)]
print("Classes:", " ".join(LETTERS))

## 2. Load & Explore the Dataset

The **A-Z Handwritten Alphabets** dataset is a CSV where each row is one
28x28 grayscale character flattened into 784 pixel columns, preceded by an
integer label (`0=A` ... `25=Z`).

In [ ]:
CSV_PATH = "A_Z Handwritten Data.csv"   # expected in the project root

# The file has no header row: column 0 is the label, columns 1..784 are pixels.
df = pd.read_csv(CSV_PATH, header=None)
df.rename(columns={0: "label"}, inplace=True)

print(f"Dataset shape : {df.shape[0]:,} samples  x  {df.shape[1]} columns")
print(f"Pixels/sample : {df.shape[1] - 1}  (28 x 28)")
print(f"Classes       : {df['label'].nunique()}")
df.head()

In [ ]:
# Separate labels and pixel matrix
y_all = df["label"].astype("int32").values
X_all = df.drop("label", axis=1).values.astype("float32")

# --- Class distribution ---
counts = pd.Series(y_all).value_counts().sort_index()
dist = pd.DataFrame({
    "letter": LETTERS,
    "count":  counts.values,
    "pct":    (counts.values / counts.values.sum() * 100).round(2),
})
print("Total samples:", f"{counts.sum():,}")
print("Most common  :", LABELS[counts.idxmax()], f"({counts.max():,})")
print("Least common :", LABELS[counts.idxmin()], f"({counts.min():,})")
dist.T

In [ ]:
# --- Class balance bar chart (dark theme) ---
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(LETTERS, counts.values, color=ACCENT, edgecolor="#0F0F1A")
# Highlight the rarest class in the secondary accent colour
bars[counts.idxmin()].set_color(ACCENT2)
ax.set_title("Class Distribution: Samples per Letter", fontsize=15, fontweight="bold", pad=12)
ax.set_xlabel("Letter")
ax.set_ylabel("Number of samples")
ax.margins(x=0.01)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# --- A 5x5 grid of sample images, each annotated with its letter ---
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_all), size=25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(9, 9))
fig.suptitle("Random Handwritten Samples", fontsize=15, fontweight="bold", color=TEXT_C)
for ax, idx in zip(axes.ravel(), sample_idx):
    ax.imshow(X_all[idx].reshape(28, 28), cmap="magma")
    ax.set_title(LABELS[y_all[idx]], color=ACCENT, fontsize=14, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- One representative sample for every class A-Z (5 x 6 grid) ---
fig, axes = plt.subplots(5, 6, figsize=(12, 10))
fig.suptitle("One Sample per Class (A-Z)", fontsize=15, fontweight="bold", color=TEXT_C)
axes = axes.ravel()
for k in range(NUM_CLASSES):
    first = np.argmax(y_all == k)            # first occurrence of class k
    axes[k].imshow(X_all[first].reshape(28, 28), cmap="bone")
    axes[k].set_title(LABELS[k], color=ACCENT, fontsize=13, fontweight="bold")
    axes[k].axis("off")
for k in range(NUM_CLASSES, len(axes)):       # hide the 4 empty cells
    axes[k].axis("off")
plt.tight_layout()
plt.show()

## 3. Preprocessing

1. Normalise pixels to **0-1**.
2. Reshape to **(28, 28, 1)** for the CNN.
3. **One-hot** encode the 26 labels.
4. **Stratified 80/10/10** train / validation / test split.
5. Configure **`ImageDataGenerator`** augmentation on the training set.

In [ ]:
# 1. Normalise to [0, 1] and 2. reshape to (28, 28, 1)
X = (X_all / 255.0).reshape(-1, 28, 28, 1)

# 3. One-hot encode labels (26 classes)
y = to_categorical(y_all, NUM_CLASSES)

print("X shape:", X.shape, "| y shape:", y.shape)
print("Pixel range:", float(X.min()), "->", float(X.max()))

In [ ]:
# 4. Stratified 80 / 10 / 10 split.
#    First carve off 20% (-> temp), then split temp in half -> 10% val / 10% test.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y_all, random_state=SEED,
)
temp_labels = np.argmax(y_temp, axis=1)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=temp_labels, random_state=SEED,
)

print(f"Train: {X_train.shape[0]:,}  ({X_train.shape[0] / len(X) * 100:.0f}%)")
print(f"Val  : {X_val.shape[0]:,}  ({X_val.shape[0] / len(X) * 100:.0f}%)")
print(f"Test : {X_test.shape[0]:,}  ({X_test.shape[0] / len(X) * 100:.0f}%)")

In [ ]:
# 5. Data augmentation on the training stream only.
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
)
datagen.fit(X_train)

BATCH_SIZE = 128
train_generator = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED)
print("Augmentation ready (rotation=10, zoom=0.1, shift=0.1). Batch size:", BATCH_SIZE)

## 4. Build the CNN Model

Three convolutional blocks followed by a dense classifier head. Batch
normalisation helps the network train faster and dropout keeps it from
overfitting the more common letters.

In [ ]:
def build_model():
    """Construct the A-Z handwritten character recognition CNN."""
    model = Sequential(name="HandScript_CNN")
    model.add(Input(shape=(28, 28, 1)))

    # Block 1
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.25))

    # Block 2
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.25))

    # Block 3
    model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.25))

    # Classifier head
    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(NUM_CLASSES, activation="softmax"))
    return model


model = build_model()
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()
print(f"\nTotal trainable parameters: {model.count_params():,}")

## 5. Train the Model

Trained for up to 30 epochs with three callbacks:
- **EarlyStopping** (patience 5, restores best weights),
- **ReduceLROnPlateau** (patience 3, factor 0.5),
- **ModelCheckpoint** saving the best model to `models/best_model.keras`.

In [ ]:
os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-6, verbose=1),
    ModelCheckpoint("models/best_model.keras", monitor="val_accuracy",
                    save_best_only=True, verbose=1),
]

EPOCHS = 30
steps_per_epoch = len(X_train) // BATCH_SIZE

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# --- Training & validation curves (dark theme, side by side) ---
hist = history.history
epochs_ran = range(1, len(hist["accuracy"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, hist["accuracy"], color=ACCENT, marker="o", label="Train")
ax1.plot(epochs_ran, hist["val_accuracy"], color=ACCENT2, marker="o", label="Validation")
ax1.set_title("Model Accuracy", fontsize=14, fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy"); ax1.legend(facecolor=AXES_BG)

ax2.plot(epochs_ran, hist["loss"], color=ACCENT, marker="o", label="Train")
ax2.plot(epochs_ran, hist["val_loss"], color=ACCENT2, marker="o", label="Validation")
ax2.set_title("Model Loss", fontsize=14, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend(facecolor=AXES_BG)

for ax in (ax1, ax2):
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Evaluate the Model

Final test-set metrics, a per-class classification report, a 26x26 confusion
matrix, and the most confused letter pairs with example images.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy : {test_acc * 100:.2f}%")
print(f"Test loss     : {test_loss:.4f}")

# Predictions on the test set
y_pred_prob = model.predict(X_test, batch_size=512, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

In [ ]:
# --- Per-class classification report (precision / recall / F1) ---
report = classification_report(y_true, y_pred, target_names=LETTERS, digits=4)
print(report)

In [ ]:
# --- Confusion matrix heatmap (26 x 26, dark theme) ---
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=False, cmap="magma",
    xticklabels=LETTERS, yticklabels=LETTERS,
    cbar_kws={"label": "Count"}, ax=ax,
)
ax.set_title("Confusion Matrix (A-Z)", fontsize=16, fontweight="bold", pad=12)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
# --- Top 5 misclassified (true -> predicted) pairs ---
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)

pairs = []
for t in range(NUM_CLASSES):
    for p in range(NUM_CLASSES):
        if cm_offdiag[t, p] > 0:
            pairs.append((t, p, cm_offdiag[t, p]))
pairs.sort(key=lambda x: x[2], reverse=True)
top5 = pairs[:5]

print("Top 5 misclassified pairs:")
for t, p, c in top5:
    print(f"  {LABELS[t]} -> {LABELS[p]} : {c} times")

# Show one example image for each confused pair
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
fig.suptitle("Top 5 Misclassified Pairs (True -> Predicted)",
             fontsize=14, fontweight="bold", color=TEXT_C)
for ax, (t, p, c) in zip(axes, top5):
    mask = (y_true == t) & (y_pred == p)
    example = X_test[np.argmax(mask)].reshape(28, 28)
    ax.imshow(example, cmap="magma")
    ax.set_title(f"{LABELS[t]} -> {LABELS[p]}\n({c}x)", color=ACCENT2, fontsize=12, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Predictions on Sample Images

Ten random test images with their true label, predicted label, and confidence.
Titles are **green** when the prediction is correct and **red** when wrong.

In [ ]:
rng = np.random.default_rng(SEED + 1)
pick = rng.choice(len(X_test), size=10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
fig.suptitle("Sample Predictions", fontsize=15, fontweight="bold", color=TEXT_C)
for ax, idx in zip(axes.ravel(), pick):
    probs = y_pred_prob[idx]
    pred = int(np.argmax(probs))
    true = int(y_true[idx])
    conf = probs[pred] * 100
    correct = pred == true
    ax.imshow(X_test[idx].reshape(28, 28), cmap="bone")
    ax.set_title(
        f"True: {LABELS[true]} | Pred: {LABELS[pred]}\n{conf:.1f}% confidence",
        color="#2ECC71" if correct else "#FF4D6D",
        fontsize=11, fontweight="bold",
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Save & Export

Persist the final model and the label-mapping dictionary so the **HandScript AI**
Streamlit app can load them for inference.

In [ ]:
# Save the final trained model (best weights already restored by EarlyStopping)
model.save("models/best_model.keras")
print("Saved model -> models/best_model.keras")

# Save label mapping {0: 'A', ... 25: 'Z'}
with open("models/label_map.json", "w") as f:
    json.dump(LABELS, f, indent=2)
print("Saved label map -> models/label_map.json")

# Save headline metrics for the README / app reference
metrics = {
    "train_accuracy": float(hist["accuracy"][-1]),
    "val_accuracy":   float(max(hist["val_accuracy"])),
    "test_accuracy":  float(test_acc),
    "test_loss":      float(test_loss),
    "parameters":     int(model.count_params()),
}
with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics  ->", json.dumps(metrics, indent=2))

## 9. Summary & Conclusion

### Results

| Metric | Value |
|---|---|
| Final train accuracy | `train_accuracy` (below) |
| Best validation accuracy | `val_accuracy` (below) |
| **Test accuracy** | `test_accuracy` (below) |
| Total parameters | `parameters` (below) |

### What the confusion matrix shows
Almost all of the predictions land on the diagonal, so the model does well
across the board. The mistakes it does make are not spread evenly. They show up
on a few pairs of letters that look alike.

### Where it struggles and why
Most of the errors are between letters that share strokes when handwritten:
I and L, O / D / Q, U and V, and G / Q. These are hard to tell apart even for a
person when the writing is quick, and at 28x28 pixels there isn't enough detail
to separate them.

### Next steps
- A CRNN with CTC loss to read whole words instead of single letters.
- Add lowercase letters and digits to cover the full EMNIST byclass set.
- Deployment: `best_model.keras` already runs in the HandScript AI app
  (`app/app.py`). A next step would be wrapping it in a small REST API and
  exporting to ONNX or TFLite for smaller devices.

In [ ]:
# Final results table rendered from the saved metrics
results = pd.DataFrame({
    "Metric": ["Train accuracy", "Validation accuracy", "Test accuracy", "Parameters"],
    "Value": [
        f"{metrics['train_accuracy'] * 100:.2f}%",
        f"{metrics['val_accuracy'] * 100:.2f}%",
        f"{metrics['test_accuracy'] * 100:.2f}%",
        f"{metrics['parameters']:,}",
    ],
})
results